## Evaluating Synthetic Data Quality with Adaption Labs


> *"Synthetic data is only as good as your ability to measure its quality."*

You have generated synthetic data — instruction-response pairs, RAG corpora, DPO preference sets — to fine-tune or evaluate your model. The uncomfortable truth is that **synthetic data can be subtly broken**: repetitive, hallucinated, off-distribution, or simply not an improvement over what you started with. You cannot tell by eye when you have thousands of rows.

This notebook teaches you how to use **[Adaption Labs](https://adaptionlabs.ai)** to run a principled, automated quality evaluation on any synthetic dataset in minutes. By the end you will have a reproducible scoring pipeline you can attach to every generation run.

It covers the following steps:

**Ingest**: Upload files in JSONL, CSV, or Parquet, or import directly from Hugging Face and Kaggle. No reformatting, no conversion scripts.

**Preprocess and review**: The system automatically infers your data's structure and metadata: column types, suggested mappings, and an initial quality profile. Review what was detected, adjust what needs adjusting, and confirm before anything runs.

**Adapt**: Adaption Labs recipe automatically configures the best solution for your dataset. Select from a library of optimized recipes: deduplication, prompt rephrasing, reasoning trace generation, and more. Configure them for your dataset. The system shows you the estimated cost before committing. No surprises.

**Review results**: As adaption completes, review evaluation metrics alongside sample outputs. See what changed, how quality shifted, and whether the results match your expectations before moving forward.

**Download**: Export the adapted dataset in your preferred format for integration into training pipelines, evaluation harnesses, or wherever your data needs to go next.

In [1]:
# install adaption

!uv pip install adaption

Using Python 3.12.13 environment at: /usr
Resolved 14 packages in 711ms
Prepared 1 package in 67ms
Installed 1 package in 8ms
 + adaption==0.3.1


In [2]:
import os
import json
import time
import csv
import getpass
from pathlib import Path

from adaption import Adaption

# API key
# Sign Up: Get your key at https://app.adaptionlabs.ai/settings/api-keys
if "ADAPTION_API_KEY" not in os.environ:
    os.environ["ADAPTION_API_KEY"] = getpass.getpass("Enter your Adaption API key: ")

client = Adaption(api_key=os.environ["ADAPTION_API_KEY"])
print("Client ready.")

Enter your Adaption API key: ··········
Client ready.


In [3]:
# load a dataset you want to evaluate either from a local file (jsonl, etc), from Kaggle or from the Hugging Face Hub

dataset = client.datasets.create_from_huggingface(
    url="https://huggingface.co/datasets/KBayoud/AdaptationLabs-synthetic-qa",
    files=["train.csv"]
)
dataset_id = dataset.dataset_id
print(f"HF dataset loaded. ID: {dataset_id}")

HF dataset loaded. ID: 5587b3d6-1e45-4dab-8d2e-b08f9b446375


In [4]:
# Start an evaluation job to evaluate the quality of the dataset.

run = client.datasets.run(
    dataset_id,
    column_mapping={
        "prompt": "instruction",  # required — your prompt column
        "completion": "response",  # optional — completion column
    },
    job_specification={
        "max_rows": 1,
    },
)
print(run.run_id)
print(f"==> run.estimated_credits_consumed: {run.estimated_credits_consumed}")

dataset-5587b3d6-1e45-4dab-8d2e-b08f9b446375-1778883000584
==> run.estimated_credits_consumed: 1.0


In [5]:
# you can check the list of the dataset you submitted for eval.
for dataset in client.datasets.list():
    print(f"{dataset.dataset_id} — {dataset.status}")

5587b3d6-1e45-4dab-8d2e-b08f9b446375 — running


In [8]:
import time
from adaption import DatasetTimeout

POLL_INTERVAL = 5  # seconds

timeout = 600
elapsed = 0

while elapsed < timeout:
    status = client.datasets.get("5587b3d6-1e45-4dab-8d2e-b08f9b446375")
    print(f"[{elapsed}s] Status: {status.status}")

    if status.status in ("succeeded", "failed"):
        if status.error:
            print(f"Error: {status.error.message}")
        break

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    print(f"Timed out after {timeout}s (last status: {status.status})")


[0s] Status: running
[5s] Status: running
[10s] Status: running
[15s] Status: running
[20s] Status: running
[25s] Status: running
[30s] Status: running
[35s] Status: running
[40s] Status: running
[45s] Status: running
[50s] Status: running
[55s] Status: running
[60s] Status: running
[65s] Status: running
[70s] Status: running
[75s] Status: running
[80s] Status: running
[85s] Status: running
[90s] Status: running
[95s] Status: running
[100s] Status: running
[105s] Status: running
[110s] Status: running
[115s] Status: running
[120s] Status: running
[125s] Status: running
[130s] Status: running
[135s] Status: running
[140s] Status: running
[145s] Status: running
[150s] Status: running
[155s] Status: running
[160s] Status: running
[165s] Status: running
[170s] Status: running
[175s] Status: running
[180s] Status: running
[185s] Status: running
[190s] Status: running
[195s] Status: running
[200s] Status: running
[205s] Status: running
[210s] Status: running
[215s] Status: running
[220s] Sta

In [9]:
try:
    final = client.datasets.wait_for_completion(dataset_id, timeout=60)
    print(f"Finished: {final.status}")
    if final.error:
        raise RuntimeError(final.error.message)
except DatasetTimeout:
    print("Timed out — poll datasets.get or get_status in your environment")


Timed out — poll datasets.get or get_status in your environment


#### After an adaptation run finishes successfully, the platform can produce evaluation signals—scores and related metrics that summarize how the augmented data compares to the original on quality dimensions the pipeline measures.

In [10]:

evaluation = client.datasets.get_evaluation(dataset_id)

print(evaluation.status)  # pending | running | succeeded | failed | skipped
if evaluation.quality:
    print(f"Score before: {evaluation.quality.score_before}")
    print(f"Score after:  {evaluation.quality.score_after}")
    print(f"Improvement:  {evaluation.quality.improvement_percent}%")

None


Download your dataset to check you dataset after the quality upgrade

In [ ]:
url = client.datasets.download(dataset_id)
print(f"Download: {url}")